# Validación final · 20/20 oficial y 20/20 equipo

**La nueva ejecución completa obtuvo 40/40 aciertos**, sin errores de ejecución.
Se generaron las 40 respuestas desde cero con el prompt reformulado; no se
reutilizaron respuestas de campañas anteriores. Se verificaron **28/28 citas
completas**, normalizando espacios y mayúsculas. Coste del agente: **0.0473 USD**,
sin incluir los embeddings de las consultas.

[agente_v2.py](agente_v2.py) contiene **80 líneas, una sola función y cero hooks
propios**. Mantiene DeepSeek/DeepInfra, Voyage, los 1.749 embeddings cacheados del
corpus, Qdrant, los filtros y el contexto por sección. Límite nativo: 6 llamadas.

Las 7 celdas de código están ejecutadas y guardadas sin errores. Qdrant quedó
detenido, con los datos conservados. Campaña:
`resultados/estudio_v002/v2_simple_6944a9ccd5b5f92e/`.

Run All requiere Docker y `OPENROUTER_API_KEY` fuera del notebook. Las respuestas
nuevas tienen coste cloud; los checkpoints permiten reanudar esta misma versión.


## Aspectos que verificamos

- **Salida estructurada:** revisamos especialmente `of-001`, `of-004` y
  `of-005`, que fallaban al finalizar en la v1.
- **Importes y unidades:** el número completo permanece en la unidad base;
  las escalas abreviadas se reservan para la explicación.
- **Comparaciones:** se consultan ambos ejercicios y se busca respaldo textual.
  La cifra principal representa el valor final; la variación se explica aparte.
- **Citas:** verificamos el fragmento atribuido y la transcripción completa,
  incluidos los signos tipográficos.

Los cambios se concentran en las instrucciones del agente. Se mantienen los
modelos, la caché del corpus, la base de datos y los criterios de evaluación.
No se añaden reparaciones automáticas ni se modifican las respuestas evaluadas.
La v1 y sus resultados se conservan en sus archivos para permitir comparaciones.

Los conjuntos son conocidos de desarrollo; esta comprobación no es un hold-out.


In [1]:
import os, sys, json, hashlib, ast
from pathlib import Path
inicio = Path.cwd().resolve()
RAIZ = next(p for p in (inicio, *inicio.parents) if (p / "src/taller_nlp").is_dir())
for p in (RAIZ, RAIZ / "src"):
    sys.path.insert(0, str(p))
os.environ.update(LANGSMITH_TRACING="false", LANGCHAIN_TRACING_V2="false")
import pandas as pd
from IPython.display import display
from experimentos.jchulvi import agente, agente_v2, qdrant_local
from taller_nlp import ManifiestoExperimento, CasoGolden
from taller_nlp.citas import cargar_fragmentos, normalizar_texto
from taller_nlp.hashing import calcular_sha256
assert os.getenv("OPENROUTER_API_KEY"), "Carga OPENROUTER_API_KEY fuera del notebook."
assert Path(agente_v2.__file__).resolve().is_relative_to(RAIZ)
print("Agente:", agente_v2.__file__)
print("Modelo:", agente.MODELO_CLOUD, "| embeddings:", agente.MODELO_EMBEDDINGS)


Agente: /Users/jchulvi/projects/Taller_NLP-jchulvi-v2/experimentos/jchulvi/agente_v2.py
Modelo: openrouter:deepseek/deepseek-v4-flash-0731 | embeddings: voyageai/voyage-4-lite


## 1. Misma infraestructura y menos código

Se reutiliza el índice del corpus; no se recalculan los 1.749 vectores. Las consultas
nuevas usan Voyage, igual que en v1. El límite nativo pasa de 18 a 6 llamadas.
`agente.py` se conserva como referencia; su antiguo middleware está desactivado.


In [2]:
indice = agente.preparar_embeddings_cloud()
qdrant = qdrant_local.QdrantLocal(
    agente.DIRECTORIO_RESULTADOS, ruta_indice=indice,
    corpus=agente.baseline.crear_corpus_baseline(),
    modelo=agente.MODELO_EMBEDDINGS, contrato=agente.CONTRATO_EMBEDDINGS,
)
constructor = agente_v2.crear_constructor()
assert constructor.middlewares == ()
assert constructor.configuracion.modelo == agente.MODELO_CLOUD
assert constructor.configuracion.max_iteraciones == 6
assert constructor.fabrica_herramientas.retriever.parametros["almacen"] == "qdrant"
fuente = Path(agente_v2.__file__).read_text()
funciones = [x.name for x in ast.walk(ast.parse(fuente)) if isinstance(x, ast.FunctionDef)]
assert funciones == ["crear_constructor"]
assert not any(isinstance(x, ast.ClassDef) for x in ast.walk(ast.parse(fuente)))
print("Funciones:", funciones, "| líneas:", len(fuente.splitlines()), "| middlewares propios: 0")
print("Índice:", indice.name, "| dimensión:", qdrant.metadatos["dimension"])
print("SHA256 vectores:", qdrant.metadatos["sha256_vectores"])
print("Colección Qdrant:", qdrant.coleccion)


Funciones: ['crear_constructor'] | líneas: 80 | middlewares propios: 0
Índice: embeddings_cloud_92113b6e8fbccc47 | dimensión: 1024
SHA256 vectores: e9d702ecfe822c485eba2ad9eac7fb54685905ce5d523adeb28c54b2fb006c40
Colección Qdrant: jchulvi_cloud_92113b6e8fbccc47_e9d702ecfe822c48


## 2. Una pasada por conjunto

Los fallos se conservan. El identificador de campaña depende del código, los datos
y el índice: una modificación no reutiliza automáticamente respuestas de otra versión.
El agente recibe solamente la pregunta. El evaluador ve las respuestas esperadas.


In [3]:
DATASETS = {"oficial": RAIZ / "golden_set_oficial.jsonl", "equipo": RAIZ / "golden_set.jsonl"}
archivos = [*sorted((RAIZ / "src/taller_nlp").glob("*.py")),
            RAIZ / "experimentos/baseline.py", Path(agente.__file__),
            Path(agente_v2.__file__), Path(qdrant_local.__file__)]
identidad = {
    "codigo": {str(p.relative_to(RAIZ)): calcular_sha256(p) for p in archivos},
    "golden": {k: calcular_sha256(p) for k, p in DATASETS.items()},
    "vectores": qdrant.metadatos["sha256_vectores"],
    "corpus": [constructor.corpus.sha256_chunks, constructor.corpus.sha256_xbrl],
    "configuracion": constructor.configuracion.model_dump(mode="json"),
}
huella = hashlib.sha256(json.dumps(identidad, sort_keys=True).encode()).hexdigest()[:16]
CAMPANA = agente.DIRECTORIO_RESULTADOS / ("v2_simple_" + huella)
CAMPANA.mkdir(parents=True, exist_ok=True)
(CAMPANA / "configuracion.json").write_text(json.dumps(identidad, indent=2))
CASOS = {k: CasoGolden.cargar_jsonl(p, constructor.corpus, numero_esperado=20) for k, p in DATASETS.items()}
informes = {}
print("Campaña:", CAMPANA)


def evaluar(conjunto):
    variante = constructor.con_ruta_progreso(CAMPANA / f"{conjunto}.progreso.json")
    with qdrant.sesion():
        informe = variante.construir().evaluar(DATASETS[conjunto])
    ManifiestoExperimento.desde_constructor(variante, informe=informe,
        metadatos=identidad).guardar(CAMPANA / f"{conjunto}.manifiesto.json")
    print(f"{conjunto}: {informe.aciertos_totales}/{informe.numero_preguntas}")
    return informe


Campaña: /Users/jchulvi/projects/Taller_NLP-jchulvi-v2/experimentos/jchulvi/resultados/estudio_v002/v2_simple_6944a9ccd5b5f92e


In [4]:
informes["oficial"] = evaluar("oficial")

oficial: 20/20


In [5]:
informes["equipo"] = evaluar("equipo")

equipo: 20/20


## 3. Resultado y regresión de los tres fallos

La nota usa el evaluador común (protocolo 5). Se muestra además la literalidad de
**toda** la cita: el evaluador solo verifica su prefijo normalizado de 120 caracteres.
Esta comprobación adicional no modifica ni repara ninguna respuesta.


In [6]:
chunks = cargar_fragmentos(constructor.corpus.ruta_chunks)
filas = []
for conjunto, informe in informes.items():
    for r in informe.resultados:
        a = r.respuesta_agente
        completa = (any(cid in chunks and normalizar_texto(a.cita) in normalizar_texto(chunks[cid].texto)
                        for cid in a.citas) if a.cita else None)
        filas.append({"conjunto": conjunto, "id": r.id_pregunta, "acierto": r.acierto,
                      "cita_completa": completa, "cifra": r.cifra_correcta,
                      "trayectoria": r.trayectoria_correcta, "error": a.error,
                      "coste_usd": a.coste_usd, "segundos": a.latencia_ms / 1000,
                      "herramientas": len(a.llamadas)})
tabla = pd.DataFrame(filas)
resumen = tabla.groupby("conjunto").agg(evaluadas=("id", "size"), aciertos=("acierto", "sum"),
    errores=("error", "count"), coste_agente_usd=("coste_usd", lambda s: s.sum(min_count=len(s))),
    latencia_mediana_s=("segundos", "median"), latencia_p95_s=("segundos", lambda s: s.quantile(.95)))
display(resumen)
display(tabla[tabla.id.isin(["of-001", "of-004", "of-005"])])
display(tabla)
tabla.to_csv(CAMPANA / "resultados.csv", index=False)
print("Objetivo 20/20 en ambos:", bool((resumen.aciertos == 20).all()))
print("Qdrant detenido:", not qdrant.en_marcha)


,evaluadas,aciertos,errores,coste_agente_usd,latencia_mediana_s,latencia_p95_s
conjunto,,,,,,
equipo,20,20,0,0.026319,17.428207,42.557298
oficial,20,20,0,0.021013,16.198790,68.853318


,conjunto,id,acierto,cita_completa,cifra,trayectoria,error,coste_usd,segundos,herramientas
0,oficial,of-001,True,True,None,True,None,0.001462,15.331032,2
3,oficial,of-004,True,True,None,True,None,0.001200,49.250865,2
4,oficial,of-005,True,True,None,True,None,0.001259,8.582367,2


,conjunto,id,acierto,cita_completa,cifra,trayectoria,error,coste_usd,segundos,herramientas
0,oficial,of-001,True,True,None,True,None,0.001462,15.331032,2
1,oficial,of-002,True,True,None,True,None,0.001058,68.246850,2
2,oficial,of-003,True,True,None,True,None,0.002294,20.200807,2
3,oficial,of-004,True,True,None,True,None,0.001200,49.250865,2
4,oficial,of-005,True,True,None,True,None,0.001259,8.582367,2
5,oficial,of-006,True,True,None,True,None,0.000541,6.740805,2
6,oficial,of-007,True,None,True,True,None,0.000539,33.130824,3
7,oficial,of-008,True,True,True,True,None,0.001340,18.996251,4
8,oficial,of-009,True,None,True,True,None,0.000368,28.158336,2
9,oficial,of-010,True,None,True,True,None,0.000385,6.240774,2


Objetivo 20/20 en ambos: True
Qdrant detenido: True


## 4. Respuestas y evidencia para revisión

Estos son conjuntos conocidos de desarrollo, no una evaluación ciega. Un 20/20
mide los criterios del evaluador; no garantiza corrección semántica de cada frase.
Se conserva el detalle para revisar esa diferencia. El coste del agente excluye
los embeddings de consulta.


In [7]:
for conjunto, informe in informes.items():
    preguntas = {c.id: c for c in CASOS[conjunto]}
    for r in informe.resultados:
        a = r.respuesta_agente
        print(f"\n{'=' * 70}\n{conjunto} · {r.id_pregunta} · {'ACIERTO' if r.acierto else 'FALLO'}")
        print("PREGUNTA:", preguntas[r.id_pregunta].pregunta)
        print("REFERENCIA:", preguntas[r.id_pregunta].respuesta_esperada)
        print("RESPUESTA:", a.respuesta)
        print("CIFRA/UNIDAD:", a.cifra, a.unidad, "| FUENTE:", a.fuente)
        print("CITA:", a.cita, "| CHUNKS:", a.citas)
        for llamada in a.llamadas:
            print("HERRAMIENTA:", llamada.nombre, json.dumps(llamada.argumentos, ensure_ascii=False))
        print("OBSERVACIONES:", r.observaciones, "| ERROR:", a.error)



oficial · of-001 · ACIERTO
PREGUNTA: ¿Qué dice NVIDIA en su 10-K de FY2025 sobre la competencia en el mercado chino y los controles de exportación?
REFERENCIA: Que el mercado chino, donde sus productos están limitados por los controles de exportación, es muy competitivo y espera que lo siga siendo.
RESPUESTA: En su 10-K de FY2025 (Item 1A, Factores de Riesgo), NVIDIA señala que el mercado chino, donde su oferta está limitada por los controles de exportación, es muy competitivo y prevé que siga siéndolo. Además, describe que los controles de exportación de EE. UU. (incluida la IFR "AI Diffusion" de enero de 2025) han perjudicado su posición competitiva: restringen el envío de productos como A100, H100, H200, B200 y GB200 a China y a más de 150 países "Tier 2", no ha recibido licencias para exportar esos productos restringidos a China, y advierte que tales controles pueden beneficiar a competidores (incluidos proveedores de semiconductores de China, Europa e Israel) y empujar a los clie